# Civic Connect - GPU Training Notebook
Run this on **Google Colab** (Runtime → Change runtime type → T4 GPU)
Trains both the vision-only model and the multimodal fusion model.

In [ ]:
# @title 1. Mount Drive & Clone Repo
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content')
!git clone https://github.com/Yash913212/Civic-Connect.git civic-connect
os.chdir('civic-connect/backend')

In [ ]:
# @title 2. Install Dependencies
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install transformers sentencepiece scikit-learn pillow numpy<2 einops accelerate

In [ ]:
# @title 3. Train Vision Model (EfficientNetB0, 8 classes)
!python train_model.py

In [ ]:
# @title 4. Download MuRIL for Multimodal Training
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('google/muril-base-cased')
model = AutoModel.from_pretrained('google/muril-base-cased')
print('MuRIL downloaded')

In [ ]:
# @title 5. Train Multimodal Fusion Model (Vision + Text → 8 Depts + Priority)
import sys
import torch
sys.path.insert(0, '.')

from app.ai.train_multimodal import train, device, DATA_DIR

print(f'GPU available: {torch.cuda.is_available()}')
print(f'Device: {device}')

# Train with GPU-optimized config
train(data_dir=DATA_DIR, epochs=30, batch_size=32, lr=3e-4)


In [ ]:
# @title 6. Download trained models
from google.colab import files
files.download('civic_model.pth')
files.download('civic_model_multimodal.pth')

In [ ]:
# @title 7. Run Evaluation
!python evaluate.py